# SS3_PartB: Applying Logistic Regression to STADIOalot Client Data

**Chosen method:** Logistic Regression (the preferred method from SS2)  
**Primary target:** `late_delivery` (1 = late or never successfully delivered; 0 = not late)

The notebook applies the selected method to STADIOalot's supplied operational extracts. It uses one record per order and information that would be available by the time the first delivery route is being planned. The aim is to estimate which dispatched orders are at greater risk of late delivery.


## Dataset loading

The supplied extract contains six CSV tables. The order table contains the outcome and order details; the route, customer-history, driver/vehicle and operational tables provide features at delivery-attempt level. The data-quality table is used for data-quality context, it is not as a source of predictive features.

Tables 2–5 can contain several rows per order because an order may have more than one delivery attempt. To avoid giving an order extra weight or using later attempts to predict its final outcome, this notebook takes the **earliest recorded route attempt per order**. Cancelled orders with no dispatch or route attempt and rows with a missing target are excluded because they do not represent an order that reached the delivery-planning stage.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, RocCurveDisplay, PrecisionRecallDisplay
)

RANDOM_STATE = 42

# Find the provided CSVs, assuming data is contained in of the candidate paths, relative to root folder.
candidate_dirs = [
    Path("datasets/stadioalot_client_data"),
    Path("client-data"),
    Path("."),
    Path("datasets/client-data"),
    Path("../client-data"),
]
data_dir = next((p for p in candidate_dirs if (p / "1_orders_deliveries.csv").exists()), None)
if data_dir is None:
    raise FileNotFoundError(
        "Could not find 1_orders_deliveries.csv. Put all six CSV files in a "
        "'client-data' folder beside this notebook, or update candidate_dirs."
    )

orders = pd.read_csv(data_dir / "1_orders_deliveries.csv")
route = pd.read_csv(data_dir / "2_delivery_route.csv")
customer = pd.read_csv(data_dir / "3_customer_delivery_history.csv")
driver = pd.read_csv(data_dir / "4_driver_vehicle.csv")
operational = pd.read_csv(data_dir / "5_operational_time.csv")
quality = pd.read_csv(data_dir / "6_data_quality_reference.csv")

tables = {
    "Orders": orders,
    "Delivery route / attempts": route,
    "Customer history": customer,
    "Driver and vehicle": driver,
    "Operational time": operational,
    "Data quality reference": quality,
}
pd.DataFrame(
    [{"Table": name, "Rows": df.shape[0], "Columns": df.shape[1]} for name, df in tables.items()]
)

In [ ]:
# Normalise text fields and key IDs to reduce casing/whitespace mismatches.
def clean_text_columns(df):
    df = df.copy()
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].map(lambda x: x.strip() if isinstance(x, str) else x)
    return df

orders, route, customer, driver, operational, quality = [
    clean_text_columns(df) for df in
    [orders, route, customer, driver, operational, quality]
]

# Parse dates robustly, including mixed date formats in the raw extract.
date_columns = {
    "orders": ["order_date", "dispatch_date", "promised_delivery", "delivery_date"],
    "route": ["route_start"],
    "quality": ["record_timestamp"],
}
for col in date_columns["orders"]:
    orders[col] = pd.to_datetime(orders[col], errors="coerce", format="mixed", dayfirst=True)
route["route_start"] = pd.to_datetime(route["route_start"], errors="coerce", format="mixed", dayfirst=True)
quality["record_timestamp"] = pd.to_datetime(quality["record_timestamp"], errors="coerce", format="mixed", dayfirst=True)

# Standardise target values and retain one row per order.
orders["late_delivery"] = pd.to_numeric(orders["late_delivery"], errors="coerce")
orders["repeat_delivery"] = pd.to_numeric(orders["repeat_delivery"], errors="coerce")
orders = orders.drop_duplicates(subset=["order_id"], keep="first").copy()
orders = orders[orders["late_delivery"].isin([0, 1])].copy()

print(f"Orders after de-duplication and removing missing/invalid target: {len(orders):,}")
print("Late-delivery class counts:")
display(orders["late_delivery"].value_counts().rename(index={0: "Not late (0)", 1: "Late / unsuccessful (1)"}).to_frame("Orders"))
print("Class proportions:")
display((orders["late_delivery"].value_counts(normalize=True).sort_index() * 100).rename(index={0: "Not late (0)", 1: "Late / unsuccessful (1)"}).round(2).to_frame("Percent"))

### Target balance and scope

The target is `late_delivery`, as specified in the supplied README and carried forward from SS2. The notebook reports the observed class counts and proportions after duplicate handling and removal of missing/invalid target labels. Because the target classes are not equally represented, Logistic Regression uses `class_weight='balanced'` so the less common late-delivery class is not ignored during training.

The target in the file also includes orders that were never successfully delivered. However, cancelled orders with no dispatch/attempt information and no target label cannot be learned from at the chosen prediction point, so they are excluded from this model run. The separate `repeat_delivery` field is not used as the target in this notebook.

## Dataset preprocessing

#### Joining and defining the prediction point

The model predicts the final late-delivery outcome using information available by the **first recorded route attempt / route-planning stage**. It joins the earliest route row to the order and then joins the matching point-in-time customer, driver/vehicle and operational records using `delivery_id`.

Fields only known after an attempt—such as actual delivery date, actual distance/duration, delivery status/outcome, final attempt count and return reason—are deliberately excluded. This prevents the model from learning from information that would not have been available when a risk estimate was needed.

In [ ]:
# Use the earliest route_start per order as the first route attempt.
route_sorted = route.sort_values(["order_id", "route_start", "delivery_id"], na_position="last")
first_route = route_sorted.drop_duplicates(subset=["order_id"], keep="first").copy()
first_route = first_route.drop(columns=["actual_distance_km", "actual_duration_minutes"], errors="ignore")

# Join the first-attempt delivery_id to point-in-time history and operational features.
data = orders.merge(first_route, on="order_id", how="inner", suffixes=("", "_route"))
data = data.merge(customer.drop(columns=["order_id", "customer_id"], errors="ignore"),
                  on="delivery_id", how="left")
data = data.merge(driver.drop(columns=["driver_id", "vehicle_id"], errors="ignore"),
                  on="delivery_id", how="left")
data = data.merge(operational.drop(columns=["order_id"], errors="ignore"),
                  on="delivery_id", how="left")

# Remove orders without dispatch information; the stated use case is delivery planning
# for orders that have entered the dispatch/delivery process.
data = data[data["dispatch_date"].notna()].copy()
data = data[data["late_delivery"].isin([0, 1])].copy()

# Convert date columns from strings to datetime
date_columns = [
    "order_date",
    "dispatch_date",
    "promised_delivery",
    "route_start"
]

for col in date_columns:
    data[col] = pd.to_datetime(
        data[col],
        errors="coerce",
        dayfirst=False
    )

# Calculate date-based features
data["order_to_dispatch_hours"] = (
    data["dispatch_date"] - data["order_date"]
).dt.total_seconds() / 3600

# Derive features from dates known at the chosen prediction point.
data["order_to_dispatch_hours"] = (
    (data["dispatch_date"] - data["order_date"]).dt.total_seconds() / 3600
)
data["promised_lead_hours"] = (
    (data["promised_delivery"] - data["dispatch_date"]).dt.total_seconds() / 3600
)
data["order_day_of_week"] = data["order_date"].dt.day_name()
data["dispatch_day_of_week"] = data["dispatch_date"].dt.day_name()
data["dispatch_month"] = data["dispatch_date"].dt.month
data["route_start_hour"] = data["route_start"].dt.hour

# Remove impossible derived values caused by bad timestamps; mark them missing.
for col in ["order_to_dispatch_hours", "promised_lead_hours"]:
    data.loc[(data[col] < 0) | (data[col] > 24 * 90), col] = np.nan

print(f"Modeling rows (one per eligible order): {len(data):,}")
print(f"Unique orders: {data['order_id'].nunique():,}")
print(f"Unique delivery IDs selected: {data['delivery_id'].nunique():,}")
print("Target distribution:")
display(data["late_delivery"].value_counts().sort_index().rename(index={0: "Not late (0)", 1: "Late / unsuccessful (1)"}).to_frame("Orders"))

#### Cleaning, feature selection and data splitting

- **Duplicate orders:** duplicate `order_id` rows are reduced to one order record.
- **Inconsistent text:** leading/trailing spaces are removed; categorical features are encoded consistently by the preprocessing pipeline.
- **Missing values:** numeric missing values are filled with the median and categorical missing values with the most frequent value. Missingness is expected in optional route/weather feeds.
- **Implausible values:** impossible or very large negative/positive date intervals are treated as missing rather than used as valid durations.
- **Identifiers and leakage:** IDs are not used as predictive features. Post-outcome fields are not used. Data-quality cancellation/return reasons are also excluded because they can reveal outcomes.
- **Chronological split:** data is sorted by order date and split into 70% training, 15% validation and 15% final test. This better reflects future use than a random split. The validation set is used to choose the probability threshold; the final test set remains unseen until evaluation.

In [ ]:
# Select features known by route planning / first attempt.
numeric_features = [
    "order_value", "item_count",
    "order_to_dispatch_hours", "promised_lead_hours",
    "route_distance_km", "planned_duration_minutes", "stop_number",
    "customer_tenure_months", "previous_orders", "previous_failures",
    "previous_late_deliveries", "previous_address_issues",
    "driver_experience_months", "driver_deliveries",
    "driver_late_rate", "driver_failed_rate",
    "vehicle_capacity_kg", "vehicle_load_kg",
    "public_holiday", "peak_period", "orders_on_route",
    "orders_per_driver", "warehouse_processing_time_minutes",
    "delivery_latitude", "delivery_longitude", "dispatch_month", "route_start_hour"
]
categorical_features = [
    "product_category", "delivery_type", "dc_id", "delivery_area",
    "delivery_window", "traffic_condition", "weather_condition",
    "preferred_delivery_time", "address_type", "vehicle_type",
    "day_of_week", "order_day_of_week", "dispatch_day_of_week"
]

# Remove feature columns not present in a particular version of the extracts.
numeric_features = [c for c in numeric_features if c in data.columns]
categorical_features = [c for c in categorical_features if c in data.columns]
feature_columns = numeric_features + categorical_features

# Sort by order date to ensure the latest records are reserved for final testing.
data = data.sort_values("order_date", na_position="first").reset_index(drop=True)
X = data[feature_columns].copy()
y = data["late_delivery"].astype(int).copy()

n = len(data)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_val, y_val = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
X_test, y_test = X.iloc[val_end:], y.iloc[val_end:]

split_summary = pd.DataFrame({
    "Subset": ["Training", "Validation", "Final test"],
    "Rows": [len(y_train), len(y_val), len(y_test)],
    "Late orders": [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
    "Late share (%)": [round(y_train.mean()*100, 2), round(y_val.mean()*100, 2), round(y_test.mean()*100, 2)]
})
display(split_summary)

## Model training

### Why Logistic Regression and these settings?

Logistic Regression was selected in SS2 because it identified substantially more late deliveries than Random Forest on the earlier Olist evaluation. It is also relatively straightforward to explain to a business audience and provides a probability score that can be used to prioritise follow-up.

- **`class_weight='balanced'`:** compensates for the difference in the number of late and non-late orders.
- **`C=1.0`:** a moderate regularisation setting; it limits overly large coefficients without imposing unusually strong restrictions.
- **`solver='liblinear'`:** a reliable solver for binary classification with a one-vs-rest-compatible setup.
- **`max_iter=1000`:** allows sufficient iterations for the model to converge after preprocessing.
- **Scaling and encoding:** numeric features are standardised, while categorical features are one-hot encoded. This is important because the source includes numeric measures and categories such as delivery area, weather and vehicle type.

This is a classification model, not a neural network, so learning rate and epochs do not apply. The fitted preprocessing and model are kept together in one pipeline to ensure the same transformations are applied consistently to validation and test data.

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        C=1.0,
        solver="liblinear",
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

model.fit(X_train, y_train)
print("Logistic Regression trained.")
print(f"Features: {len(feature_columns)} ({len(numeric_features)} numeric, {len(categorical_features)} categorical)")

## Model validation

The validation set is used to select a practical probability threshold. The default threshold of 0.50 is not automatically the best business choice when late deliveries are less common and missing a late order can be costly. Here, the threshold that gives the best validation F1-score is selected from candidate thresholds. F1 balances precision and recall; STADIOalot should later confirm the threshold against the real cost of unnecessary checks versus missed late deliveries.

After the threshold is selected, the model is assessed once on the later, unseen test set. Test results provide the more realistic estimate of performance on future orders.

In [ ]:
val_prob = model.predict_proba(X_val)[:, 1]
candidate_thresholds = np.arange(0.10, 0.91, 0.01)

threshold_rows = []
for threshold in candidate_thresholds:
    val_pred = (val_prob >= threshold).astype(int)
    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(y_val, val_pred, zero_division=0),
        "recall": recall_score(y_val, val_pred, zero_division=0),
        "f1": f1_score(y_val, val_pred, zero_division=0)
    })
threshold_table = pd.DataFrame(threshold_rows)
best_row = threshold_table.loc[threshold_table["f1"].idxmax()]
chosen_threshold = float(best_row["threshold"])

print(f"Chosen threshold from validation set: {chosen_threshold:.2f}")
display(best_row.to_frame("Validation metric"))

In [ ]:
def calculate_metrics(y_true, probabilities, threshold, subset_name):
    predictions = (probabilities >= threshold).astype(int)
    return {
        "Subset": subset_name,
        "Rows": len(y_true),
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(y_true, predictions, zero_division=0),
        "Recall": recall_score(y_true, predictions, zero_division=0),
        "F1-score": f1_score(y_true, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, probabilities) if y_true.nunique() == 2 else np.nan,
        "PR-AUC (Average Precision)": average_precision_score(y_true, probabilities) if y_true.nunique() == 2 else np.nan
    }

train_prob = model.predict_proba(X_train)[:, 1]
test_prob = model.predict_proba(X_test)[:, 1]

metrics = pd.DataFrame([
    calculate_metrics(y_train, train_prob, chosen_threshold, "Training"),
    calculate_metrics(y_val, val_prob, chosen_threshold, "Validation"),
    calculate_metrics(y_test, test_prob, chosen_threshold, "Final unseen test")
]).set_index("Subset")

display(metrics.round(4))
print("Final unseen test classification report:")
print(classification_report(y_test, (test_prob >= chosen_threshold).astype(int),
                            target_names=["Not late (0)", "Late / unsuccessful (1)"],
                            zero_division=0))

In [ ]:
# Confusion matrix for the final unseen test set.
test_pred = (test_prob >= chosen_threshold).astype(int)
cm = confusion_matrix(y_test, test_pred, labels=[0, 1])

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm)
ax.set_title(f"Final test confusion matrix (threshold={chosen_threshold:.2f})")
ax.set_xlabel("Predicted class")
ax.set_ylabel("Actual class")
ax.set_xticks([0, 1], ["Not late", "Late"])
ax.set_yticks([0, 1], ["Not late", "Late"])
for (i, j), value in np.ndenumerate(cm):
    ax.text(j, i, str(value), ha="center", va="center")
fig.colorbar(im, ax=ax, label="Number of orders")
plt.tight_layout()
plt.show()